# The Template Pattern - Forcing a Structured Response

Hello everyone. Whenever interacting with LLMs, it will very often be the case that the model's reasoning and its final answer are mixed together in a free-form block of text, making it difficult for our programs to use the output reliably.

This is where a highly practical and robust technique comes in: **The Template Pattern.**

The core idea is profound: instead of letting the model decide the structure of its response, **we force the model to fill in a predefined template.** This gives us complete control over the entire output, making it perfectly and predictably parsable.

This notebook will demonstrate how to use this pattern to separate the model's "thinking" from its final "answer."

## The Task and Helper Functions

Our task will be to refactor a piece of Python code. We want the model to first *think* about the necessary changes and then provide the final code. We will compare a standard Chain-of-Thought response with a templated one.

In [18]:
import inspect
import litellm
from IPython.display import display, Markdown
from textwrap import dedent
#from dotenv import load_dotenv

#load_dotenv()

MODEL_NAME = "ollama/qwen2.5:3b"
MAX_TOKENS_DEFAULT = 500

# --- The "messy" code to be refactored ---
def get_user_initials(user_data_list):
    initials_list = []
    for user in user_data_list:
        if 'name' in user and len(user['name']) > 0:
            parts = user['name'].split(' ')
            initials = ''
            for part in parts:
                initials += part[0].upper()
            initials_list.append(initials)
    return initials_list

CODE_TO_REFACTOR = inspect.getsource(get_user_initials)

def get_completion(
    prompt,
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS_DEFAULT,
    **kwargs
):
    parsed_messages = []

    if type(prompt) is str:
        parsed_messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        parsed_messages = prompt

    response = litellm.completion(
        model=model,
        messages=parsed_messages,
        max_tokens=max_tokens,
        **kwargs
    )

    return response.choices[0].message.content

print("Setup complete. Helper functions and code context are ready.")

Setup complete. Helper functions and code context are ready.


## Unstructured Chain-of-Thought

Let's start by using a standard Chain-of-Thought prompt. We'll ask the model to think step-by-step and then provide the code. Watch how the reasoning and the final code are intertwined in the output.

In [19]:
prompt_unstructured_cot = [
    {
        "role": "system",
        "content": "You are a senior Python developer who refactors code to be more idiomatic and readable"
    },
    {
        "role": "user",
        "content": dedent(f"""
        Please refactor the following Python code.
        Let's think step-by-step to plan changes and then provide the refactored code.

        ```python
        {CODE_TO_REFACTOR}
        ```
        """)              
    }
]
output_unstructured_cot = get_completion(prompt_unstructured_cot)

In [20]:
display(Markdown(output_unstructured_cot))

Certainly! Let's refactor the given code step-by-step. The goal is to make the code more readable, maintainable, and idiomatic. Here's a breakdown of the changes we'll make:

1. **Use a List Comprehension:** We can replace the loop with a list comprehension, which will make the code more concise.
2. **Use a Regular Expression for Name Validation:** Instead of manually checking if `'name'` exists and if the length of `user['name']` is greater than zero, we can use a regular expression to ensure the presence of a name.
3. **Improve Error Handling:** We'll add a simple error handling mechanism if `user['name']` is not a string or if it's an empty string.

Let's start with the refactored code:

```python
import re

def get_user_initials(user_data_list):
    initials_list = []
    for user in user_data_list:
        if 'name' in user and isinstance(user['name'], str) and len(user['name']) > 0:
            initials = re.sub(r'\W+', '', user['name']).upper()[0]
            initials_list.append(initials)
        else:
            initials_list.append('')
    return initials_list
```

### Explanation:

1. **List Comprehension:** We're using a list comprehension to generate the `initials_list` in a more concise manner.
2. **Regular Expression for Name Validation:** The `re.sub(r'\W+', '', user['name'])` function replaces all non-word characters (`\W+`) in the name with an empty string, ensuring we get only the first letter of the first name in uppercase.
3. **Error Handling:** The condition `if 'name' in user and isinstance(user['name'], str) and len(user['name']) > 0:` checks that the user data contains a `'name'` key, the value is a string, and the length is greater than zero.

This code should be more robust and readable while retaining the same functionality.

## The Template Pattern

Now, we'll solve this problem by providing a strict template. We will define clear sections using custom tags like `[THOUGHTS]` and `[REFACTORED_CODE]` and instruct the model to place its response *only* within these sections.

In [15]:
response_template = dedent("""
[THOUGHTS START]
Place your step-by-step reasoning and analysis of the code here
[THOUGHTS END]

[REFACTORED CODE START]
Place only the final, complete, and refactored Python code here.
Do not use backticks.
[REFACTORED CODE END]

""")

prompt_with_template = [
    {
        "role": "system",
        "content": "You are a senior Python developer who refactors code to be more idiomatic and readable"
    },
    {
        "role": "user",
        "content": dedent(f"""
        Please refactor the following Python code.
        Let's think step-by-step to plan changes and then provide the refactored code.

        You must structure your response following the template below.
        Do not provide any additional text or output outside of the provided tags.

        ### Template:
        {response_template}
        
        <python_code>
        {CODE_TO_REFACTOR}
        </python_code>
        """)              
    }
]
output_structured = get_completion(prompt_with_template)

In [17]:

print(output_structured)

### Template:

[THOUGHTS START]
The given function `get_user_initials` takes a list of user data, each containing a `name` key, and extracts initials from the names. The function iterates over each user, checks for the presence of a name, and then splits the name into parts. It then concatenates the first character of each part into a string. The function appends this string to a list of initials. Finally, it returns the list of initials.

To make the code more idiomatic and readable, we can:
1. Use a list comprehension for a more concise approach.
2. Use `zip` to iterate through the parts and names simultaneously, which can eliminate the need for an explicit loop.
3. Utilize `str.title` method to automatically capitalize the first letter of a string.
[THOUGHTS END]

[REFACTORED CODE START]
def get_user_initials(user_data_list):
    return [part[0].upper() for user in user_data_list if 'name' in user and len(user['name']) > 0
            for part in user['name'].split(' ')]

[REFACTORED CODE END]

### Template:

[THOUGHTS START]
The given function `get_user_initials` takes a list of user data, each containing a `name` key, and extracts initials from the names. The function iterates over each user, checks for the presence of a name, and then splits the name into parts. It then concatenates the first character of each part into a string. The function appends this string to a list of initials. Finally, it returns the list of initials.

To make the code more idiomatic and readable, we can:
1. Use a list comprehension for a more concise approach.
2. Use `zip` to iterate through the parts and names simultaneously, which can eliminate the need for an explicit loop.
3. Utilize `str.title` method to automatically capitalize the first letter of a string.
[THOUGHTS END]

[REFACTORED CODE START]
def get_user_initials(user_data_list):
    return [part[0].upper() for user in user_data_list if 'name' in user and len(user['name']) > 0
            for part in user['name'].split(' ')]

[REFACTORE